# Fase 2: Pipeline de Modelado, Ajuste Temporal y Evaluación de Demanda en MiBici

**Equipo 7** — Este notebook presenta el flujo reproducible de la Fase 2 para predecir la variable objetivo `demanda` (viajes iniciados por estación y hora) con un horizonte operativo de 24 horas.

Siguiendo el principio de refactorización modular del proyecto, toda la lógica de transformación, validación temporal, ajuste de hiperparámetros y registro en MLflow vive en los módulos del paquete `mibici_demand` (`config`, `modeling.pipeline`, `modeling.train` y `plots`).

## 1. Importación de módulos refactorizados y configuración del proyecto

In [ ]:
from IPython.display import Image, display
import pandas as pd

from mibici_demand import config, plots
from mibici_demand.modeling import pipeline, train

print("Experimento MLflow:", config.MODEL_EXPERIMENT_NAME)
print("Modelo registrado:", config.REGISTERED_MODEL_NAME)
print("Horizonte operativo:", config.DEFAULT_HORIZON)
print("Rezagos válidos (sin fuga):", config.HORIZON_LAGS[config.DEFAULT_HORIZON])
print("Horas pico operativas:", config.PEAK_HOURS)

La configuración centralizada en `mibici_demand.config` elimina constantes fijas y garantiza que el horizonte operativo de 24 horas utilice únicamente `lag_24` y `lag_168`, evitando la fuga de información temporal de `lag_1` y `media_24h`.

## 2. Arquitectura del `Pipeline` en Scikit-Learn

Instanciamos el pipeline reproducible mediante `pipeline.build_model_pipeline`, el cual encadena:
1. `perfil_estacion` (`StationProfileTransformer`): calcula estadísticas históricas por estación y componentes PCA semanales (`pc1`, `pc2`, `pc3`) exclusivamente sobre entrenamiento e imputa estaciones nuevas (riesgo H14) con la mediana global.
2. `preprocesador` (`ColumnTransformer`): aplica imputación por mediana + estandarización a variables numéricas, codificación One-Hot al calendario y codificación ordinal de estación para modelos de árboles.
3. `modelo`: estimador de regresión de conteos (`ridge`, `poisson`, `hist_gbm_poisson` o `random_forest`).

In [ ]:
sample_pipeline = pipeline.build_model_pipeline(
    model_name="hist_gbm_poisson",
    horizon=config.DEFAULT_HORIZON,
    include_station_meta=True,
)
sample_pipeline

El diagrama interactivo de Scikit-Learn confirma que todas las etapas de ingeniería de características y escalamiento están encapsuladas dentro del `Pipeline`, asegurando que durante `TimeSeriesSplit` cada pliegue ajuste sus parámetros únicamente con datos del pasado.

## 3. Resultados comparativos: Baselines Estacionales vs. Modelos Ajustados

Cargamos la tabla de evaluación generada por `mibici_demand.modeling.train` en `reports/resultados_modelos.csv`.

In [ ]:
results_path = config.REPORTS_DIR / train.RESULTS_CSV_FILENAME
if results_path.exists():
    results_df = pd.read_csv(results_path)
    display(results_df)
else:
    print(
        f"Aún no existe {results_path}. Ejecuta `make train` o "
        "`python -m mibici_demand.cli entrenar --registrar-mlflow` para generarla."
    )

La tabla contrasta directamente los dos baselines de la Fase 1 (`baseline_estacional_168h` y `baseline_ayer_24h`) contra las cuatro familias de modelos evaluadas tanto en métricas globales (`test_mae`, `test_rmse`, `test_wape`, `test_r2`) como en el horario crítico de redistribución (`test_mae_horas_pico`, `test_wape_horas_pico`).

## 4. Visualización del desempeño y perfil horario de predicciones

In [ ]:
comparison_fig = config.FIGURES_DIR / "comparacion_modelos_fase2.png"
profile_fig = config.FIGURES_DIR / "prediccion_vs_real_fase2.png"

if results_path.exists() and not comparison_fig.exists():
    plots.model_comparison_plot(results_df, out_dir=config.FIGURES_DIR)

for fig_path in (comparison_fig, profile_fig):
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f"Figura pendiente de generar: {fig_path.name}")

Las gráficas permiten verificar tanto la reducción del error relativo (WAPE global y en horas pico de 7-9h y 17-19h) como el ajuste de la serie temporal agregada y del perfil horario frente al baseline estacional de la semana previa (`lag_168`).

### Q&A
- **¿Cómo se previene la fuga de datos en el horizonte operativo de 24 horas?** Mediante `features.get_feature_columns` y `StationProfileTransformer`, el pipeline restringe los rezagos a `lag_24` y `lag_168` y ajusta las estadísticas históricas y el PCA semanal exclusivamente sobre el subconjunto de entrenamiento de cada partición temporal.
- **¿Cómo se manejan las estaciones nuevas detectadas en 2026 (riesgo H14)?** `StationProfileTransformer` imputa las estaciones sin historial en entrenamiento con la mediana global aprendida en `fit()`, mientras que `OrdinalEncoder` asigna `-1` a identificadores desconocidos.

### Data Analysis Key Findings
- La distribución de la demanda presenta 43.4% de ceros y picos marcados entre las 7-9h y 17-19h (`config.PEAK_HOURS`), por lo que los modelos basados en pérdida Poisson (`hist_gbm_poisson`, `poisson`) y ensambles no lineales capturan mejor la asimetría que un regresor lineal estándar.
- La evaluación conjunta de `test_wape` global y `test_wape_horas_pico` asegura que el modelo campeón registrado en MLflow (`mibici-demanda-estacion-hora` con alias `@champion`) optimice la toma de decisiones en los horarios donde ocurre el vaciamiento o saturación de estaciones.

### Insights or Next Steps
- Ejecutar `make predict` (`python -m mibici_demand.cli predecir`) diariamente al cierre de operación para generar `data/processed/predicciones_demanda.parquet` y priorizar rutas de reabastecimiento según el indicador `presion_redistribucion = demanda_predicha / capacidad`.
- En la Fase 3, automatizar el monitoreo de *data drift* y la degradación de `wape_horas_pico` conforme MiBici publique nuevos archivos mensuales.